In [1]:
import importlib
import sys
import torch
import numpy as np

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')

In [2]:
import event_log_loader.new_event_log_loader_v2
importlib.reload(event_log_loader.new_event_log_loader_v2)
from event_log_loader.new_event_log_loader_v2 import PrefixesDataFrameLoader, EventLogLoader, EventLogPerturbationPreprocess

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(17)

event_log_location = '../../../../../../data/data/helpdesk.csv'

result_name = 'helpdesk_all'

cat_dynamic = ['Activity', 'Resource']
# cat_static =  ['VariantIndex', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup']
cat_static =  ['seriousness', 'product', 'responsible_section', 'service_level', 'service_type', 'support_section', 'workgroup']

num_dynamic = ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day']
num_dyn_log = []
num_static = []

event_log_properties = {# case id
                        'case_name' : 'CaseID',
                        # activit
                        'concept_name' : 'Activity',
                        # time values and computaitons
                        'timestamp_name' : 'CompleteTimestamp',
                        'date_format' : '%Y/%m/%d %H:%M:%S.%f',
                        'time_since_case_start_column' : 'case_elapsed_time',
                        'time_since_last_event_column' : 'event_elapsed_time',
                        'day_in_week_column' : 'day_in_week',
                        'seconds_in_day_column' : 'seconds_in_day',
                        # min suffix size for eos padding right
                        'min_suffix_size' : 5,
                        # trian and test split
                        'train_validation_size' : 0.15,
                        'test_validation_size' : 0.2,
                        # window size for padding
                        'window_size' : 'auto',
                        # dynamic and static values
                        'categorical_columns' : cat_dynamic,
                        'static_categorical_columns' : cat_static,
                        'continuous_columns' : num_dynamic,
                        'continuous_positive_columns' : num_dyn_log,
                        'static_continuous_columns' : num_static}


In [3]:
# object to create datframe of prefixes for petri-net repaly marking computation
pref_adopt_dataframe = PrefixesDataFrameLoader(event_log_location=event_log_location, event_log_properties=event_log_properties)

In [4]:
raw_data = pref_adopt_dataframe.get_raw_dataframe()
raw_data

,CaseID,Activity,Resource,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,seriousness,product,responsible_section,service_level,service_type,support_section,workgroup
0,Case 1,"[Assign seriousness, Take in charge ticket, Ta...","[Value 1, Value 1, Value 2, Value 1, Value 3]","[0.0, 44.0, 259959.0, 1371849.0, 2671462.0]","[nan, 44.0, 259915.0, 1111890.0, 1299613.0]","[1, 1, 4, 3, 4]","[53417, 53461, 54176, 42866, 46479]",Value 1,Value 1,Value 1,Value 1,Value 1,Value 1,Value 1
1,Case 10,"[Assign seriousness, Take in charge ticket, Re...","[Value 2, Value 2, Value 2, Value 5]","[0.0, 3196606.0, 3196613.0, 4489038.0]","[nan, 3196606.0, 7.0, 1292425.0]","[2, 4, 4, 5]","[31820, 31626, 31633, 28058]",Value 1,Value 3,Value 2,Value 3,Value 1,Value 2,Value 3
2,Case 100,"[Assign seriousness, Take in charge ticket, Re...","[Value 1, Value 9, Value 9, Value 2, Value 3]","[0.0, 1036724.0, 1056354.0, 2861743.0, 4157763.0]","[nan, 1036724.0, 19630.0, 1805389.0, 1296020.0]","[4, 2, 2, 2, 3]","[37517, 37441, 57071, 48060, 48080]",Value 1,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1
3,Case 1000,"[Assign seriousness, Assign seriousness, Take ...","[Value 2, Value 2, Value 2, Value 2, Value 5]","[0.0, 6.0, 12.0, 450805.0, 3802049.0]","[nan, 6.0, 6.0, 450793.0, 3351244.0]","[3, 3, 3, 1, 5]","[32008, 32014, 32020, 50813, 32457]",Value 1,Value 3,Value 4,Value 2,Value 1,Value 3,Value 1
4,Case 1001,"[Assign seriousness, Take in charge ticket, Re...","[Value 4, Value 4, Value 4, Value 3]","[0.0, 1436829.0, 1704597.0, 2997012.0]","[nan, 1436829.0, 267768.0, 1292415.0]","[4, 0, 3, 4]","[61284, 29313, 37881, 34296]",Value 1,Value 2,Value 1,Value 2,Value 1,Value 1,Value 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4575,Case 995,"[Assign seriousness, Take in charge ticket, Wa...","[Value 8, Value 8, Value 6, Value 6, Value 6, ...","[0.0, 30.0, 90286.0, 3194141.0, 3194763.0, 449...","[nan, 30.0, 90256.0, 3103855.0, 622.0, 1296011.0]","[2, 2, 3, 4, 4, 5]","[31711, 31741, 35597, 29052, 29674, 29685]",Value 1,Value 3,Value 1,Value 2,Value 2,Value 1,Value 2
4576,Case 996,"[Assign seriousness, Take in charge ticket, Re...","[Value 8, Value 6, Value 6, Value 5]","[0.0, 325091.0, 1885211.0, 3181227.0]","[nan, 325091.0, 1560120.0, 1296016.0]","[3, 0, 4, 5]","[49903, 29394, 34314, 34330]",Value 1,Value 3,Value 1,Value 2,Value 2,Value 1,Value 2
4577,Case 997,"[Assign seriousness, Take in charge ticket, Re...","[Value 1, Value 13, Value 13, Value 5]","[0.0, 258637.0, 871435.0, 2851523.0]","[nan, 258637.0, 612798.0, 1980088.0]","[4, 0, 0, 2]","[48912, 48349, 56347, 49235]",Value 1,Value 2,Value 1,Value 2,Value 1,Value 1,Value 1
4578,Case 998,"[Assign seriousness, Take in charge ticket, Wa...","[Value 9, Value 2, Value 9, Value 9, Value 3]","[0.0, 25.0, 338294.0, 1745662.0, 3041677.0]","[nan, 25.0, 338269.0, 1407368.0, 1296015.0]","[4, 4, 1, 3, 4]","[43033, 43058, 35727, 60695, 60710]",Value 1,Value 1,Value 1,Value 1,Value 1,Value 1,Value 1


In [5]:
train_pref_df = pref_adopt_dataframe.get_dataset('train')
train_pref_df

,CaseID,prefix_length,Activity,Resource,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,seriousness,product,responsible_section,service_level,service_type,support_section,workgroup
0,Case 10,1,[Assign seriousness],[Value 2],[0.0],[nan],[2.0],[31820.0],Value 1,Value 3,Value 2,Value 3,Value 1,Value 2,Value 3
1,Case 10,2,"[Assign seriousness, Take in charge ticket]","[Value 2, Value 2]","[0.0, 3196606.0]","[nan, 3196606.0]","[2.0, 4.0]","[31820.0, 31626.0]",Value 1,Value 3,Value 2,Value 3,Value 1,Value 2,Value 3
2,Case 10,3,"[Assign seriousness, Take in charge ticket, Re...","[Value 2, Value 2, Value 2]","[0.0, 3196606.0, 3196613.0]","[nan, 3196606.0, 7.0]","[2.0, 4.0, 4.0]","[31820.0, 31626.0, 31633.0]",Value 1,Value 3,Value 2,Value 3,Value 1,Value 2,Value 3
3,Case 100,1,[Assign seriousness],[Value 1],[0.0],[nan],[4.0],[37517.0],Value 1,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1
4,Case 100,2,"[Assign seriousness, Take in charge ticket]","[Value 1, Value 9]","[0.0, 1036724.0]","[nan, 1036724.0]","[4.0, 2.0]","[37517.0, 37441.0]",Value 1,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10936,Case 993,4,"[Assign seriousness, Take in charge ticket, Wa...","[Value 8, Value 8, Value 2, Value 2]","[0.0, 235.0, 1200632.0, 1379474.0]","[nan, 235.0, 1200397.0, 178842.0]","[0.0, 0.0, 0.0, 2.0]","[40283.0, 40518.0, 31315.0, 37357.0]",Value 1,Value 2,Value 1,Value 2,Value 1,Value 1,Value 1
10937,Case 993,5,"[Assign seriousness, Take in charge ticket, Wa...","[Value 8, Value 8, Value 2, Value 2, Value 2]","[0.0, 235.0, 1200632.0, 1379474.0, 1379486.0]","[nan, 235.0, 1200397.0, 178842.0, 12.0]","[0.0, 0.0, 0.0, 2.0, 2.0]","[40283.0, 40518.0, 31315.0, 37357.0, 37369.0]",Value 1,Value 2,Value 1,Value 2,Value 1,Value 1,Value 1
10938,Case 996,1,[Assign seriousness],[Value 8],[0.0],[nan],[3.0],[49903.0],Value 1,Value 3,Value 1,Value 2,Value 2,Value 1,Value 2
10939,Case 996,2,"[Assign seriousness, Take in charge ticket]","[Value 8, Value 6]","[0.0, 325091.0]","[nan, 325091.0]","[3.0, 0.0]","[49903.0, 29394.0]",Value 1,Value 3,Value 1,Value 2,Value 2,Value 1,Value 2


In [6]:
event_log_loader = EventLogLoader(event_log_location=event_log_location, event_log_properties=event_log_properties, prefix_df=pref_adopt_dataframe)

In [7]:
train_dataset = event_log_loader.get_dataset('train')
torch.save(train_dataset, '../../../../encoded_data/'+result_name+'_'+str(event_log_loader.encoder_decoder.min_suffix_size)+'_train.pkl')
train_dataset

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

Activity:   0%|          | 0/2977 [00:00<?, ?it/s]

Resource:   0%|          | 0/2977 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/4 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

event_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/2977 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/2977 [00:00<?, ?it/s]

In [8]:
print("Categories and encoding params for dynamic categorical and continuous values: ", train_dataset.all_categories)
print("Categories and encoding params for static categorical and continuous values: ", train_dataset.all_static_categories)

print("Dynamic categorical tensor: ", train_dataset.categorical_tensors[0].size())
print("Dynamic continuous tensor: ", train_dataset.continuous_tensors[0].size())

print("Static categorical tensor: ", train_dataset.static_categorical_tensor.size())
print("Static continuous tensor: ", train_dataset.static_continuous_tensor.size())

print("Zero padding tensor: ", train_dataset.zero_padding.size())
print("EOS padding tensor: ",train_dataset.eos_padding.size())
# print(train_dataset.zero_padding[0:5])
# print(train_dataset.eos_padding[0:5])

print("Empty prefix petri net replay tensor: ", train_dataset.prefixes_petri_net_marking.size())

Categories and encoding params for dynamic categorical and continuous values:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Categories and encoding params for static categorical and continuous values:  ([('seriousness', 2, {'Value 1': 1}), ('product'

In [9]:
perturb_preprocess = EventLogPerturbationPreprocess(event_log_location=event_log_location, event_log_properties=event_log_properties)
train_df_pert, val_df_pert, test_df_pert = perturb_preprocess.get_all_datasets()
train_df_pert

,CaseID,Activity,Resource,CompleteTimestamp,VariantIndex,seriousness,customer,product,responsible_section,seriousness_2,service_level,service_type,support_section,workgroup,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day
9,Case 10,Assign seriousness,Value 2,2010-02-10 08:50:20,1.0,Value 1,Value 10,Value 3,Value 2,Value 1,Value 3,Value 1,Value 2,Value 3,0.0,NaN,2.0,31820.0
10,Case 10,Take in charge ticket,Value 2,2010-03-19 08:47:06,1.0,Value 1,Value 10,Value 3,Value 2,Value 1,Value 3,Value 1,Value 2,Value 3,3196606.0,3196606.0,4.0,31626.0
11,Case 10,Resolve ticket,Value 2,2010-03-19 08:47:13,1.0,Value 1,Value 10,Value 3,Value 2,Value 1,Value 3,Value 1,Value 2,Value 3,3196613.0,7.0,4.0,31633.0
12,Case 10,Closed,Value 5,2010-04-03 07:47:38,1.0,Value 1,Value 10,Value 3,Value 2,Value 1,Value 3,Value 1,Value 2,Value 3,4489038.0,1292425.0,5.0,28058.0
13,Case 10,EOS,EOS,NaT,NaN,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39663,Case 999,Closed,Value 3,2013-03-29 16:24:45,1.0,Value 1,Value 92,Value 3,Value 1,Value 2,Value 2,Value 2,Value 1,Value 4,3889088.0,1296015.0,4.0,59085.0
39664,Case 999,EOS,EOS,NaT,NaN,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,NaN,NaN,NaN,NaN
39665,Case 999,EOS,EOS,NaT,NaN,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,NaN,NaN,NaN,NaN
39666,Case 999,EOS,EOS,NaT,NaN,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,EOS,NaN,NaN,NaN,NaN


In [10]:
features = perturb_preprocess.extract_feature_info()